## Índice maestro del Notebook

0. Portada y objetivo del notebook  
1. Carga de Librerias y Datos (Code) 
2. Análisis Crítico de Columnas y Data Leakage

    * Paso 1: Definición de Variables (X, y) y Prevención de Data Leakage

    * Paso 2: División del Dataset (Train / Test Split)
3. Diseño y Construcción de la Tubería de Preprocesamiento

    * Paso 3: Identificación y Clasificación de Variables (Numéricas vs Categóricas)

    * Paso 4: Configuración de los Transformadores (ColumnTransformer)

4. Ensamblado, Entrenamiento y Exportación

    * Paso 5: Ensamblado del Pipeline y Entrenamiento del Modelo Base
    
    * Paso 6: Exportación y Almacenamiento del Pipeline Definitivo (.joblib)

---
 
# PREPROCESAMIENTO Y PREPARACIÓN DEL MODELO

En este cuaderno nos encargamos de transformar el conjunto de datos limpio y unificado proveniente del `01_eda_limpieza.ipynb`. 

El objetivo es preparar los "ingredientes" matemáticos perfectos para que nuestro modelo de Machine Learning (ML) pueda aprender de forma óptima sin cometer errores de software ni sesgos estadísticos.

### Objetivos Clave:
1. **Definir el Target (y)** y las **Variables Predictoras (X)**.
2. **Identificar y eliminar columnas de riesgo** para evitar el *Data Leakage* (Fuga de datos).
3. **Dividir de forma estricta** los datos en conjuntos de Entrenamiento (*Train*) y Prueba (*Test*).
4. **Construir un Pipeline automatizado** con un `ColumnTransformer` que gestione de forma robusta la imputación de valores nulos (NaN), el escalado de números y la codificación de categorías.
5. **Exportar el Pipeline entrenado en un archivo .joblib** sirviendo como el entregable definitivo y automatizado para el resto del equipo y el futuro despliegue de la aplicación.

---

# 1. CARGA DE LIBRERIAS Y DATOS (CODE)

Importamos las librerías necesarias y cargamos el dataset generado con `01_eda_limpieza.ipynb`

> **DATASET UTILIZADO:** `data/processed/station_status_history_2022_modeling_base_clean.csv`

In [23]:
# ==============================================================================
# 1. CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================

# Librerías básicas de manipulación de datos PARA TODO EL PROYECTO

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer              # Para solucionar los NaN
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.pipeline import Pipeline as SubPipeline
from sklearn.linear_model import Ridge                 # Nuestro modelo base
import joblib

# ==============================================================================

# Cargamos el dataset del EDA
df = pd.read_csv('../data/processed/station_status_history_2022_modeling_base_clean.csv')

# Le decimos a Pandas que no tenga límite al mostrar columnas
pd.set_option('display.max_columns', None)

# Visualizamos las primeras filas para verificar que todo está correcto
df.head()

,snapshot_datetime,station_id_historical,station_name,station_address,station_latitude,station_longitude,station_capacity,num_docks_available,num_bikes_available,reservations_count,is_active,is_not_available,occupation_ratio,snapshot_hour,snapshot_day_of_week,snapshot_month,year,day_of_week_name_es,day_type,is_working_day,is_weekend,weather_temperature_mean_c,weather_temperature_min_c,weather_temperature_max_c,weather_precipitation_mm,weather_humidity_mean_pct,weather_station_count,weather_temperature_available_count,weather_precipitation_available_count,weather_humidity_available_count,weather_has_precipitation
0,2022-01-01,1,Puerta del Sol A,Puerta del Sol nº 1,40.417214,-3.701834,30,0,0,0,1,1,0.000000,0,5,1,2022,sábado,festivo,False,True,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
1,2022-01-01,2,Puerta del Sol B,Puerta del Sol nº 1,40.417313,-3.701603,30,0,0,0,1,1,0.000000,0,5,1,2022,sábado,festivo,False,True,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
2,2022-01-01,3,Miguel Moya,Calle Miguel Moya nº 1,40.420589,-3.705842,24,16,7,0,1,0,0.291667,0,5,1,2022,sábado,festivo,False,True,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
3,2022-01-01,4,Plaza Conde Suchil,Plaza del Conde del Valle de Súchil nº 3,40.430294,-3.706917,18,1,14,0,1,0,0.777778,0,5,1,2022,sábado,festivo,False,True,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
4,2022-01-01,5,Malasaña,Calle Manuela Malasaña nº 5,40.428552,-3.702587,24,2,17,0,1,0,0.708333,0,5,1,2022,sábado,festivo,False,True,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False


In [24]:
# ==============================================================================
# CELDA DE DIAGNÓSTICO: AUDITORÍA DE COLUMNAS METEOROLÓGICAS (AEMET)
# ==============================================================================

# 1. Filtramos todas las columnas que empiecen por "weather_" o "clima"
weather_cols = [col for col in df.columns if 'weather' in col.lower() or 'clima' in col.lower()]

print(f"🔍 Se han encontrado {len(weather_cols)} columnas meteorológicas en el dataset:")
print(weather_cols)
print("-" * 80)

if len(weather_cols) > 0:
    # 2. Analizamos cuántos nulos (NaN) reales tienen estas columnas
    print("📊 1. CONTROL DE HUECOS (¿Están rellenas o vacías?):")
    nulos_weather = df[weather_cols].isnull().sum()
    pct_nulos_weather = (df[weather_cols].isnull().mean()) * 100
    
    for col in weather_cols:
        print(f"   • {col}: {nulos_weather[col]:,} nulos ({pct_nulos_weather[col]:.2f}%)")
    print("-" * 80)
    
    # 3. Inspeccionamos un resumen estadístico rápido para ver si los valores tienen sentido
    print("📈 2. RESUMEN ESTADÍSTICO RÁPIDO (¿Tienen coherencia los datos?):")
    display(df[weather_cols].describe(include='all'))
    print("-" * 80)
    
    # 4. Sacamos una pequeña muestra aleatoria para ver qué aspecto real tienen
    print("👀 3. MUESTRA ALEATORIA DE FILAS RELLENAS:")
    display(df[weather_cols].sample(5, random_state=42))

else:
    print("🚨 ¡ALERTA! No se ha detectado ninguna columna que contenga la palabra 'weather' o 'clima'.")

🔍 Se han encontrado 10 columnas meteorológicas en el dataset:
['weather_temperature_mean_c', 'weather_temperature_min_c', 'weather_temperature_max_c', 'weather_precipitation_mm', 'weather_humidity_mean_pct', 'weather_station_count', 'weather_temperature_available_count', 'weather_precipitation_available_count', 'weather_humidity_available_count', 'weather_has_precipitation']
--------------------------------------------------------------------------------
📊 1. CONTROL DE HUECOS (¿Están rellenas o vacías?):
   • weather_temperature_mean_c: 0 nulos (0.00%)
   • weather_temperature_min_c: 0 nulos (0.00%)
   • weather_temperature_max_c: 0 nulos (0.00%)
   • weather_precipitation_mm: 0 nulos (0.00%)
   • weather_humidity_mean_pct: 0 nulos (0.00%)
   • weather_station_count: 0 nulos (0.00%)
   • weather_temperature_available_count: 0 nulos (0.00%)
   • weather_precipitation_available_count: 0 nulos (0.00%)
   • weather_humidity_available_count: 0 nulos (0.00%)
   • weather_has_precipitation: 

,weather_temperature_mean_c,weather_temperature_min_c,weather_temperature_max_c,weather_precipitation_mm,weather_humidity_mean_pct,weather_station_count,weather_temperature_available_count,weather_precipitation_available_count,weather_humidity_available_count,weather_has_precipitation
count,2.306832e+06,2.306832e+06,2.306832e+06,2.306832e+06,2.306832e+06,2.306832e+06,2.306832e+06,2.306832e+06,2.306832e+06,2306832
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1921392
mean,1.643487e+01,9.719767e+00,2.314720e+01,7.966354e-02,4.757214e+01,4.670634e+00,4.670634e+00,4.670634e+00,4.670634e+00,NaN
std,7.253549e+00,6.537446e+00,8.224513e+00,2.558576e-01,1.485321e+01,6.190023e-01,6.190023e-01,6.190023e-01,6.190023e-01,NaN
min,6.840000e+00,2.360000e+00,1.058000e+01,0.000000e+00,2.700000e+01,3.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,NaN
25%,1.020000e+01,3.120000e+00,1.776000e+01,0.000000e+00,3.100000e+01,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,NaN
50%,1.598000e+01,8.060000e+00,2.388000e+01,0.000000e+00,5.080000e+01,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,NaN
75%,2.377500e+01,1.485000e+01,3.168000e+01,0.000000e+00,5.980000e+01,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,NaN


--------------------------------------------------------------------------------
👀 3. MUESTRA ALEATORIA DE FILAS RELLENAS:


,weather_temperature_mean_c,weather_temperature_min_c,weather_temperature_max_c,weather_precipitation_mm,weather_humidity_mean_pct,weather_station_count,weather_temperature_available_count,weather_precipitation_available_count,weather_humidity_available_count,weather_has_precipitation
2127782,6.84,3.12,10.58,0.02,42.0,5,5,5,5,True
940457,15.98,8.06,23.88,0.00,46.8,5,5,5,5,False
2118186,6.84,3.12,10.58,0.02,42.0,5,5,5,5,True
134623,10.06,2.36,17.76,0.00,63.6,5,5,5,5,False
82059,10.06,2.36,17.76,0.00,63.6,5,5,5,5,False


# 2. ANÁLISIS CRÍTICO DE COLUMNAS Y DATA LEAKAGE

## Paso 1: Definición de Variables (X, y) y Prevención de Data Leakage

En esta sección se realizará la separación fundamental del conjunto de datos para preparar el entrenamiento del modelo predictivo de **BiciMad**.

### Variable Objetivo / Target (`y`):
* **`num_bikes_available`**: Este modelo tiene como objetivo predecir el **número absoluto de bicicletas disponibles** en cada estación. Al tratarse de un valor numérico continuo, resolveremos esto como un problema de **Regresión**.

### Variables Predictoras / Características (`X`):
Para construir la matriz `X` (las pistas que usará el modelo), realizamos un filtrado estricto eliminando una *lista negra* de columnas bajo los siguientes criterios técnicos y de negocio:

1. **Prevención de Data Leakage (Fuga de datos):** Eliminamos `num_docks_available` (anclajes disponibles), `occupation_ratio` (ratio de ocupación) y `reservations_count` (total de reservas).
   * *Justificación:* Aunque estos datos existen en nuestro histórico, **no se conocerán en el futuro** cuando un usuario use la aplicación (por ejemplo, para saber la disponibilidad de mañana). Si los dejamos, el modelo "haría trampas" memorizando restas perfectas en el cuaderno en lugar de aprender los patrones reales (clima, hora, días laborables), haciendo que la aplicación final no funcione en el mundo real.
2. **Eliminación de Redundancias y Duplicados del EDA:**
   * Eliminamos `source_month`, `month` y `day_of_week` porque están duplicadas. Se decide mantener sus versiones homólogas (`snapshot_month` y `snapshot_day_of_week`).
   * Eliminamos `holiday_name` y `holiday_type` por ser demasiado específicas; el modelo aprenderá mejor usando la variable booleana general `is_holiday` (True/False).

### Variables que mantenemos para la Aplicación:
* Decidimos **mantener** variables como `station_name`, `station_address` y `day_of_week_name_es`. Aunque a nivel matemático contengan información que ya podría estar representada en los códigos numéricos de las estaciones, son **esenciales para la experiencia de usuario (UX) de nuestra futura App**, permitiendo interactuar con nombres de calles y días legibles.
* Asimismo, conservamos identificadores temporales e históricos clave como `date`, `snapshot_datetime` y `station_id_historical`. Todas estas variables serán gestionadas y transformadas adecuadamente en nuestra tubería automática como variables categóricas.

In [25]:
# 1. Definimos nuestro Target (lo que la app va a adivinar)
y = df["num_bikes_available"]

# 2. Lista negra de columnas a eliminar para X (Basada en nuestro análisis)
columnas_a_eliminar = [
    "num_bikes_available",      # Es el target (y), obligatoriamente fuera de X
    
    # --- DATA LEAKAGE (Trampas del futuro) ---
    "num_docks_available",      # No sabremos los huecos libres en el futuro
    "occupation_ratio",         # No sabremos el ratio de llenado en el futuro
    "reservations_count",       # No sabremos las reservas del futuro
    
    # --- REPETIDAS / DUPLICADAS ---
    #"source_month",             # Duplicada (nos quedamos con snapshot_month)
    #"month",                    # Duplicada (nos quedamos con snapshot_month)
    #"day_of_week",              # Duplicada (nos quedamos con snapshot_day_of_week)
    
    # --- REDUNDANCIAS DE FESTIVOS ---
    #"holiday_name",             # Demasiado específico
    #"holiday_type"              # Redundante con 'is_holiday'
]

# 3. Creamos las variables predictoras X manteniendo las columnas clave para la app
X = df.drop(columns=columnas_a_eliminar)

# 4. Comprobamos las dimensiones finales en la pizarra de juego
print(f"Dimensiones de X (Pistas para el modelo): {X.shape}")
print(f"Dimensiones de y (Objetivo a predecir): {y.shape}")

Dimensiones de X (Pistas para el modelo): (2306832, 27)
Dimensiones de y (Objetivo a predecir): (2306832,)


## Paso 2: División del Dataset (Train / Test Split)

Evaluamos de forma objetiva el rendimiento de nuestro modelo de Regresión, para ello dividimos los datos en dos subconjuntos independientes:

1. **Conjunto de Entrenamiento (Train - 80%):** Los datos que el modelo utilizará para descubrir las relaciones entre el clima, las horas, las estaciones y la disponibilidad de bicicletas.
2. **Conjunto de Prueba (Test - 20%):** Un conjunto de datos que guardamos "bajo llave". El modelo nunca los verá durante su aprendizaje. Nos servirá como el "examen final" para comprobar cuántas bicicletas de diferencia falla el modelo en condiciones reales.


> *Nota de control:* Fijamos un `random_state=42` para garantizar que la división de los datos sea siempre la misma cada vez que ejecutemos el cuaderno (reproducibilidad).

In [26]:
# Dividimos los datos: 80% para entrenar y 20% para testear
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)

# Comprobamos los tamaños de los nuevos conjuntos
print(f"Datos de Entrenamiento (X_train): {X_train.shape} | (y_train): {y_train.shape}")
print(f"Datos de Examen (X_test): {X_test.shape} | (y_test): {y_test.shape}")

Datos de Entrenamiento (X_train): (1845465, 27) | (y_train): (1845465,)
Datos de Examen (X_test): (461367, 27) | (y_test): (461367,)


# 3. DISEÑO Y CONSTRUCCIÓN DE LA TUBERIA DE PREPROCESAMIENTO

## Paso 3: Identificación y Clasificación de Variables (Numéricas vs Categóricas)

Para que el **Pipeline** sepa qué engranaje de preprocesamiento aplicar a cada columna, clasificamos las 32 variables de la matriz `X_train` en dos grandes grupos:

1. **Variables Numéricas (`numerical_cols`):** Información puramente cuantitativa como el clima (temperatura, velocidad del viento, precipitación) o componentes temporales indexados como números (año, hora). Estas variables pasarán por un proceso de **Estandarización**.
2. **Variables Categóricas (`categorical_cols`):** Información cualitativa o etiquetas de texto (nombre de la estación, dirección, nombre del día de la semana) y variables booleanas/ID que representan categorías (número de estación, si es festivo o día laborable). Estas variables pasarán por un proceso de **Codificación (One-Hot Encoding)**.

### 🛡️ Programación Defensiva: Control de Consistencia

En lugar de confiar a ciegas en la lista manual de columnas, implementamos en el código un **chivato de seguridad matemático**: Este sistema suma dinámicamente la cantidad de variables asignadas a ambos sacos y las contrasta con las columnas reales de `X_train`.

Si la diferencia es exactamente **0**, el cuaderno nos da luz verde; si se nos hubiera olvidado alguna columna o hubiéramos arrastrado "variables fantasma" del pasado, el sistema daría la alarma inmediatamente, evitando que entren datos corruptos al modelo.

In [27]:
# 1. DEFINIMOS LAS COLUMNAS NUMÉRICAS (Limpias de variables inexistentes)
numerical_cols = [
    # Temporales básicas
    "snapshot_hour",
    "year",
    
    # Datos físicos de la estación
    "station_capacity",
    "station_latitude",
    "station_longitude",
    
    # Clima (Las que de verdad existen en tu dataset)
    "weather_temperature_mean_c",
    "weather_temperature_max_c",
    "weather_temperature_min_c",
    "weather_humidity_mean_pct",
    "weather_precipitation_mm",
    
    # Conteos técnicos del clima
    "weather_station_count",
    "weather_temperature_available_count",
    "weather_humidity_available_count",
    "weather_precipitation_available_count"
]

# 2. DEFINIMOS LAS COLUMNAS CATEGÓRICAS (Manteniendo tu estructura)
categorical_cols = [
    # Identificadores y nombres de estaciones
    #"station_number",
    "station_name",
    "station_address",
    "station_id_historical",     
    
    # Fechas y días
    #"date",                      
    "snapshot_datetime",         
    "snapshot_month",          
    "snapshot_day_of_week",    
    "day_of_week_name_es",
    
    # Estados booleanos y tipos de día (True/False o Texto)
    #"is_holiday",
    "is_working_day",
    "is_active",
    "is_weekend",
    #"is_non_working_day",
    "is_not_available",
    "day_type",
    #"station_light_status",
    "weather_has_precipitation"
]

# 3. CONTROL DE SEGURIDAD INTERNO
columnas_clasificadas = numerical_cols + categorical_cols
print(f"Columnas reales en X_train: {X_train.shape[1]}")
print(f"Columnas clasificadas: {len(columnas_clasificadas)}")

if X_train.shape[1] == len(columnas_clasificadas):
    print("¡Perfecto! Todas las columnas están clasificadas correctamente.")
else:
    faltan = set(X_train.columns) - set(columnas_clasificadas)
    sobran = set(columnas_clasificadas) - set(X_train.columns)
    if faltan: print(f"⚠️ Faltan por clasificar: {faltan}")
    if sobran: print(f"⚠️ Columnas sobrantes (no están en X_train): {sobran}")

Columnas reales en X_train: 27
Columnas clasificadas: 27
¡Perfecto! Todas las columnas están clasificadas correctamente.


In [28]:
# Programación Defensiva: Control de Consistencia
set(X_train.columns) - set(numerical_cols + categorical_cols)

set()

## Paso 4: Configuración de los Transformadores (ColumnTransformer)

Una vez clasificadas nuestras variables, definimos las estrategias de preprocesamiento matemático que se aplicarán de forma automatizada. Para garantizar que el flujo sea robusto ante datos ausentes, estructuramos el preprocesamiento de cada saco mediante sub-tuberías(`Pipeline`):

1. **Para Variables Numéricas (`num_transformer`):** 
   * `SimpleImputer(strategy='median')`: *Estrategia de Imputación*. Detecta cualquier celda vacía o nula (`NaN`) en las variables cuantitativas (por ejemplo, si un día falló el sensor meteorológico y no registró la temperatura) y la rellena automáticamente con la mediana de esa columna para no alterar las propiedades estadísticas del dataset.
   * `StandardScaler()`: *Estrategia de Escalado*. Centra los datos (media = 0) y los escala según su desviación estándar. Es vital para que variables con magnitudes muy diferentes (por ejemplo, la `station_capacity` que se mueve en decenas y el year que está en los miles) no confundan al modelo, asegurando que todas tengan el mismo peso en las ecuaciones matemáticas.

2. **Para Variables Categóricas (`cat_transformer`):**
   * `SimpleImputer(strategy='most_frequent')`: *Estrategia de Imputación*. Si algún registro de texto, identificador o booleano viene vacío, el sistema lo repara en microsegundos asignándole el valor más común o frecuente de dicha columna.
   * `OneHotEncoder(handle_unknown='ignore')`: *Estrategia de Codificación*. Convierte las etiquetas de texto, fechas e IDs categóricos en matrices de unos (`1`) y ceros (`0`).
   * *Detalle de seguridad*: Configuramos `handle_unknown='ignore'`. Esto es crucial para la futura aplicación web: si mañana el Ayuntamiento de Madrid abre una nueva estación de BiciMad con un nombre o número que no existía en el histórico de entrenamiento, el Pipeline no se romperá ni tumbará el servidor; simplemente ignorará esa categoría nueva y la app seguirá prediciendo con total normalidad.

Unificamos ambos bloques en una estructura centralizada llamada `ColumnTransformer`, la cual mapea cada sub-tubería con su respectivo saco de columnas.

In [29]:
# 1. ENGRANAJE NUMÉRICO: Primero rellena nulos con la mediana, luego escala
num_transformer = SubPipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')), # ¡El reparador de nulos!
        ('scaler', StandardScaler())
    ]
)

# 2. ENGRANAJE CATEGÓRICO: Primero rellena nulos con el más frecuente, luego codifica en 1s y 0s
cat_transformer = SubPipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')), # Rellena textos vacíos
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# 3. Unificamos ambos engranajes en el preprocesador central
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

print("✅ ¡Engranajes del preprocesador actualizados con REPARADORES DE NULOS integrados!")

✅ ¡Engranajes del preprocesador actualizados con REPARADORES DE NULOS integrados!


# 4. ENSAMBLADO, ENTRENAMIENTO Y EXPORTACIÓN

## 🚀 Paso 5: Ensamblado del Pipeline y Entrenamiento del Modelo Base

Unificamos todo nuestro trabajo en el objeto definitivo: el **Pipeline de Scikit-Learn**.

1. **Ensamblado:** Conectamos de forma secuencial nuestro `preprocessor` (la máquina transformadora) con el algoritmo de Machine Learning `Ridge()` Elegimos este modelo de regresión lineal regularizado porque es estadísticamente óptimo para manejar conjuntos de datos masivos y es altamente eficiente al gestionar la matriz dispersa que genera el *One-Hot Encoding*, previniendo el sobreajuste (*overfitting*) gracias a su penalización matemática.
2. **Entrenamiento (`.fit`):** Al ejecutar el método `.fit()`, ocurre la magia: los datos en bruto de `X_train` entran en la tubería, el sistema limpia los nulos, escala los números, codifica las categorías y alimenta directamente al algoritmo Ridge para que aprenda los patrones históricos con los 1.8 millones de registros.

In [30]:


# 1. Construimos la tubería uniendo tus engranajes con el modelo matemático
pipeline_definitivo = Pipeline(
    steps=[
        ('preprocesamiento', preprocessor),   # Tu máquina del Bloque 4
        ('modelo', Ridge())                   # El algoritmo predictor
    ]
)

print("⏳ Entrenando el Pipeline completo con 1.8 millones de registros... Por favor, espera.")

# 2. Entrenamos TODO el Pipeline de golpe usando los datos de estudio
pipeline_definitivo.fit(X_train, y_train)

print("💪 ¡PIPELINE ENTRENADO CON ÉXITO!")
print("Tu máquina automatizada ya sabe predecir la disponibilidad de BiciMad.")

⏳ Entrenando el Pipeline completo con 1.8 millones de registros... Por favor, espera.
💪 ¡PIPELINE ENTRENADO CON ÉXITO!
Tu máquina automatizada ya sabe predecir la disponibilidad de BiciMad.


## Paso 6: Exportación y Almacenamiento del Pipeline Definitivo (.joblib)

Con el **Pipeline** completamente ensamblado y entrenado con éxito sobre nuestros 1.8 millones de registros, hemos alcanzado el hito final de la fase de preparación de datos.

En lugar de exportar un simple archivo `.csv` con los datos transformados (lo cual dejaría nuestro criterio de preprocesamiento "atrapado" en el pasado y rompería la futura aplicación web al recibir nuevos datos en tiempo real), **exportamos la máquina transformadora completa**.

#### 🎯 Beneficios de este enfoque:
1. **Portabilidad (Entregable para el compañero de Optimización):** Este archivo se podrá cargar con una sola línea de código (`joblib.load()`), disponiendo al instante de los reparadores de nulos, escaladores, codificadores y el modelo base ya entrenados.
2. **Preparación para Producción (Despliegue de la App):** Cuando la interfaz web reciba los datos en bruto introducidos por un usuario (ej: *"Estación Sol, viernes, 18:00h"*), este archivo `.joblib` procesará la información en milisegundos y devolverá la predicción sin riesgo de inconsistencias ni fallos de formato.

Utilizamos la librería `joblib` para congelar y almacenar nuestro objeto en la carpeta local `models/`.

In [31]:
# Guardamos tu Pipeline completo en un archivo físico .joblib
ruta_guardado = "../models/pipeline_base_bicimad_v1.joblib"
joblib.dump(pipeline_definitivo, ruta_guardado)

print(f"📦 ¡MÁQUINA COMPLETAMENTE EMPAQUETADA Y GUARDADA EN: {ruta_guardado}!")
print("Este archivo es tu entregable definitivo para el equipo.")

📦 ¡MÁQUINA COMPLETAMENTE EMPAQUETADA Y GUARDADA EN: ../models/pipeline_base_bicimad_v1.joblib!
Este archivo es tu entregable definitivo para el equipo.
